# Notebook 08 — Evaluación de Sistemas RAG

**Semana:** 8  
**Bloque:** 4.5 — Evaluación y mejora del pipeline  
**Tiempo estimado:** 3.5-4 horas

## Qué vas a construir

- Un evaluador de faithfulness basado en LLM (juez LLM)
- Un generador automático de testsets a partir de los documentos
- Un pipeline de evaluación completo que puntúa el sistema RAG de la semana 7
- Un experimento A/B: compara el pipeline básico contra el pipeline con query rewriting
- **Reto final:** cuadro de mando de evaluación que detecta preguntas problemáticas

## Prerrequisitos

Semana 7 completada. La colección `rag_docs` debe existir en `./chroma_db`.

```bash
pip install ragas langchain-openai datasets chromadb openai python-dotenv
```

---
## Celda 0 — Setup

In [ ]:
import os
import json
import time
import uuid
import datetime
import statistics
import numpy as np
from dataclasses import dataclass, field, asdict
from pathlib import Path
from dotenv import load_dotenv
from openai import OpenAI
import chromadb

load_dotenv()

cliente_openai = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))
chroma_cliente = chromadb.PersistentClient(path="./chroma_db")

MODELO_EMBEDDING  = "text-embedding-3-small"
MODELO_GENERACION = "gpt-4o-mini"
MODELO_EVALUADOR  = "gpt-4o-mini"   # mismo modelo — en producción preferirías gpt-4o

# ── Helpers de semanas anteriores ────────────────────────────────────────────

def generar_embedding(texto: str) -> list[float]:
    r = cliente_openai.embeddings.create(model=MODELO_EMBEDDING, input=texto)
    return r.data[0].embedding

def completar(
    mensajes: list[dict],
    system: str | None = None,
    temperature: float = 0.0,
    max_tokens: int = 1024
) -> str:
    msgs = ([{"role": "system", "content": system}] if system else []) + mensajes
    r = cliente_openai.chat.completions.create(
        model=MODELO_GENERACION, messages=msgs,
        temperature=temperature, max_tokens=max_tokens
    )
    return r.choices[0].message.content

def completar_json(
    mensajes: list[dict],
    system: str | None = None,
    temperature: float = 0.0
) -> dict:
    """Como completar() pero fuerza JSON mode y parsea el resultado."""
    msgs = ([{"role": "system", "content": system}] if system else []) + mensajes
    r = cliente_openai.chat.completions.create(
        model=MODELO_EVALUADOR, messages=msgs,
        temperature=temperature,
        response_format={"type": "json_object"}
    )
    return json.loads(r.choices[0].message.content)

# ── Carga la colección del notebook 07 ───────────────────────────────────────
try:
    coleccion_rag = chroma_cliente.get_collection("rag_docs")
    print(f"Colección 'rag_docs' cargada: {coleccion_rag.count()} chunks")
except Exception as e:
    print(f"ERROR: La colección 'rag_docs' no existe. Ejecuta primero el notebook 07.")
    print(f"  Detalle: {e}")

Path("evaluaciones").mkdir(exist_ok=True)
print("Setup completado.")

---
## Celda 1 — Pipeline RAG base (del notebook 07)

Reproduce el pipeline básico y el avanzado aquí para evaluar ambos.
Añade logging completo a cada consulta.

In [ ]:
UMBRAL_RELEVANCIA = 0.65

SYSTEM_RAG = """You are a technical documentation assistant.
Answer questions using ONLY the information provided in the context.
If the answer is not found in the context, say exactly: 'INSUFFICIENT_CONTEXT'
Do not use external knowledge. Be concise and factual."""


@dataclass
class TrazaRAG:
    """Registro completo de una consulta RAG, para diagnóstico y evaluación."""
    id:                str
    timestamp:         str
    pregunta:          str
    query_retrieval:   str            # puede diferir de pregunta si hay rewriting
    chunks_recuperados: list[str]
    chunks_scores:     list[float]
    chunks_fuentes:    list[str]
    respuesta:         str
    n_chunks_usados:   int
    respondida:        bool
    latencia_ms:       float
    modelo_embedding:  str
    modelo_generacion: str
    # Métricas de evaluación (se rellenan después)
    faithfulness:      float | None = None
    context_relevance: float | None = None
    answer_relevance:  float | None = None


def recuperar(
    query: str,
    coleccion: chromadb.Collection,
    k: int = 4
) -> list[dict]:
    vector = generar_embedding(query)
    resp   = coleccion.query(
        query_embeddings=[vector],
        n_results=min(k, coleccion.count()),
        include=["documents", "distances", "metadatas"]
    )
    return [
        {"texto": doc, "similitud": round(1 - dist, 4), "meta": meta}
        for doc, dist, meta in zip(
            resp["documents"][0], resp["distances"][0], resp["metadatas"][0]
        )
    ]


def construir_prompt(pregunta: str, chunks: list[dict]) -> str:
    bloques = [
        f"[Context {i} | Source: {c['meta'].get('fuente','?')} | Score: {c['similitud']:.2f}]\n{c['texto']}"
        for i, c in enumerate(chunks, 1)
    ]
    return f"CONTEXT:\n{'\n\n---\n\n'.join(bloques)}\n\nQUESTION: {pregunta}\n\nANSWER:"


def reescribir_query(pregunta: str) -> str:
    prompt = (f"Rewrite as an optimized technical search query (only the query):\n"
              f"Question: {pregunta}\nOptimized query:")
    return completar([{"role": "user", "content": prompt}], temperature=0.0).strip()


def responder_con_traza(
    pregunta: str,
    coleccion: chromadb.Collection,
    k: int = 4,
    umbral: float = UMBRAL_RELEVANCIA,
    usar_rewriting: bool = False
) -> TrazaRAG:
    """Pipeline RAG que devuelve la respuesta + trazabilidad completa."""
    t_inicio           = time.perf_counter()
    query_retrieval    = reescribir_query(pregunta) if usar_rewriting else pregunta

    chunks_todos       = recuperar(query_retrieval, coleccion, k=k)
    chunks_relevantes  = [c for c in chunks_todos if c["similitud"] >= umbral]

    if not chunks_relevantes:
        respuesta    = "INSUFFICIENT_CONTEXT"
        respondida   = False
    else:
        prompt       = construir_prompt(pregunta, chunks_relevantes)
        respuesta    = completar(
            [{"role": "user", "content": prompt}], system=SYSTEM_RAG
        )
        respondida   = "INSUFFICIENT_CONTEXT" not in respuesta

    latencia_ms = (time.perf_counter() - t_inicio) * 1000

    return TrazaRAG(
        id=str(uuid.uuid4())[:8],
        timestamp=datetime.datetime.now().isoformat(),
        pregunta=pregunta,
        query_retrieval=query_retrieval,
        chunks_recuperados=[c["texto"] for c in chunks_relevantes],
        chunks_scores=[c["similitud"] for c in chunks_relevantes],
        chunks_fuentes=[c["meta"].get("fuente", "?") for c in chunks_relevantes],
        respuesta=respuesta,
        n_chunks_usados=len(chunks_relevantes),
        respondida=respondida,
        latencia_ms=round(latencia_ms, 1),
        modelo_embedding=MODELO_EMBEDDING,
        modelo_generacion=MODELO_GENERACION,
    )


# ── Prueba rápida ─────────────────────────────────────────────────────────────
traza_prueba = responder_con_traza(
    "What are the main features of FastAPI?", coleccion_rag
)
print(f"Pregunta:   {traza_prueba.pregunta}")
print(f"Respondida: {traza_prueba.respondida}")
print(f"Chunks:     {traza_prueba.n_chunks_usados}, scores: {traza_prueba.chunks_scores}")
print(f"Latencia:   {traza_prueba.latencia_ms} ms")
print(f"Respuesta:  {traza_prueba.respuesta[:200]}...")

---
## Celda 2 — Generador de testset sintético

Genera preguntas de evaluación a partir de los propios chunks indexados.
Cada pregunta viene con su respuesta esperada (ground truth) derivada del chunk.
El resultado se guarda en JSON para reusarlo en todos los experimentos.

In [ ]:
@dataclass
class EjemploTest:
    id:               str
    pregunta:         str
    respuesta_gold:   str    # respuesta correcta conocida
    contexto_gold:    str    # chunk del que se derivó la pregunta
    fuente:           str    # artículo de Wikipedia
    tipo:             str    # factual | conceptual | comparativa


SYSTEM_GENERADOR = """You are creating evaluation questions for a RAG system.
Generate questions that can ONLY be answered using the provided text.
Avoid generic questions that any LLM could answer from training data.
Focus on specific facts, numbers, definitions, or relationships in the text."""


def generar_ejemplos_de_chunk(
    chunk: str,
    fuente: str,
    n_preguntas: int = 2
) -> list[EjemploTest]:
    """Genera n_preguntas pares (pregunta, respuesta_gold) a partir de un chunk."""
    prompt = f"""Generate {n_preguntas} evaluation question-answer pairs from this text.
Each pair should test a specific fact or concept in the text.
Mix question types: factual (what/who/when), conceptual (how/why), and comparative (difference between).

TEXT:
{chunk}

Return a JSON object with key \"pairs\" containing a list where each item has:
- \"question\": the question
- \"answer\": the expected answer (concise, 1-3 sentences)
- \"type\": \"factual\" | \"conceptual\" | \"comparative\""""

    try:
        datos = completar_json([{"role": "user", "content": prompt}], system=SYSTEM_GENERADOR)
        ejemplos = []
        for par in datos.get("pairs", []):
            ejemplos.append(EjemploTest(
                id=str(uuid.uuid4())[:8],
                pregunta=par["question"],
                respuesta_gold=par["answer"],
                contexto_gold=chunk,
                fuente=fuente,
                tipo=par.get("type", "factual")
            ))
        return ejemplos
    except Exception as e:
        print(f"  Error generando preguntas: {e}")
        return []


def construir_testset(
    coleccion: chromadb.Collection,
    n_chunks_muestrear: int = 15,
    preguntas_por_chunk: int = 2,
    ruta_cache: str = "evaluaciones/testset.json"
) -> list[EjemploTest]:
    """
    Construye un testset muestreando chunks de la colección.
    Cachea en JSON para no regenerarlo en cada experimento.
    """
    if Path(ruta_cache).exists():
        print(f"Cargando testset desde caché: {ruta_cache}")
        with open(ruta_cache) as f:
            datos = json.load(f)
        return [EjemploTest(**d) for d in datos]

    print(f"Generando testset ({n_chunks_muestrear} chunks × {preguntas_por_chunk} preguntas)...")

    # Obtiene todos los chunks de la colección
    todos = coleccion.get(include=["documents", "metadatas"])
    docs  = todos["documents"]
    metas = todos["metadatas"]

    # Filtra chunks con suficiente contenido (descarta los muy cortos)
    pares = [(d, m) for d, m in zip(docs, metas) if len(d) > 200]

    # Muestrea sin repetición
    np.random.seed(42)
    indices   = np.random.choice(len(pares), size=min(n_chunks_muestrear, len(pares)), replace=False)
    muestra   = [pares[i] for i in indices]

    testset = []
    for i, (chunk, meta) in enumerate(muestra):
        fuente = meta.get("fuente", "desconocida")
        print(f"  [{i+1}/{len(muestra)}] {fuente[:30]}...")
        ejemplos = generar_ejemplos_de_chunk(chunk, fuente, preguntas_por_chunk)
        testset.extend(ejemplos)

    # Guarda en caché
    with open(ruta_cache, "w") as f:
        json.dump([asdict(e) for e in testset], f, indent=2)

    print(f"\nTestset generado: {len(testset)} ejemplos → guardado en '{ruta_cache}'")
    return testset


testset = construir_testset(coleccion_rag, n_chunks_muestrear=15, preguntas_por_chunk=2)

# Distribución del testset
from collections import Counter
tipos = Counter(e.tipo for e in testset)
fuentes = Counter(e.fuente for e in testset)
print(f"\nDistribución por tipo:   {dict(tipos)}")
print(f"Distribución por fuente: {dict(fuentes)}")

# Muestra 3 ejemplos
print("\nEjemplos:")
for e in testset[:3]:
    print(f"  [{e.tipo}] {e.pregunta}")
    print(f"    → Gold: {e.respuesta_gold[:80]}...")

---
## Celda 3 — Evaluador LLM: faithfulness y context relevance

Construye el juez LLM que evalúa cada respuesta generada.
Faithfulness mide si la respuesta se inventa cosas; context relevance mide si los chunks recuperados son útiles.

In [ ]:
@dataclass
class ResultadoEvaluacion:
    traza_id:          str
    pregunta:          str
    respuesta:         str
    respondida:        bool
    faithfulness:      float | None    # 0-1, None si no respondida
    context_relevance: float           # media de scores de similitud
    answer_relevance:  float | None    # 0-1, estimado por LLM
    latencia_ms:       float
    notas_faithfulness: list[str] = field(default_factory=list)


SYSTEM_FAITHFULNESS = """You are a strict fact-checker evaluating whether a generated answer
is supported by the provided context. Be precise and critical."""

PROMPT_FAITHFULNESS = """Evaluate whether the answer is faithful to the context.

CONTEXT:
{contexto}

QUESTION: {pregunta}
ANSWER: {respuesta}

For each concrete factual claim in the answer, check if it is supported by the context.
Return a JSON object with:
- "claims": list of objects with {"claim": "...", "supported": true/false, "evidence": "quote or null"}
- "faithfulness_score": float 0-1 (supported_claims / total_claims)
- "verdict": "FAITHFUL" | "PARTIALLY_FAITHFUL" | "UNFAITHFUL"
- "main_issue": one sentence about the biggest faithfulness problem, or null if fully faithful"""


def evaluar_faithfulness(traza: TrazaRAG) -> dict:
    """Evalúa si la respuesta se basa solo en el contexto. Devuelve score 0-1."""
    if not traza.respondida:
        return {"faithfulness_score": None, "verdict": "NOT_ANSWERED", "claims": [], "main_issue": None}

    contexto = "\n\n---\n\n".join(traza.chunks_recuperados)
    prompt   = PROMPT_FAITHFULNESS.format(
        contexto=contexto[:3000],   # limita para no exceder contexto
        pregunta=traza.pregunta,
        respuesta=traza.respuesta
    )
    return completar_json(
        [{"role": "user", "content": prompt}],
        system=SYSTEM_FAITHFULNESS
    )


PROMPT_ANSWER_RELEVANCE = """Evaluate how relevant this answer is to the question.

QUESTION: {pregunta}
ANSWER: {respuesta}

Return JSON with:
- "relevance_score": float 0-1 (0=completely irrelevant, 1=perfectly relevant)
- "reason": one sentence explanation"""


def evaluar_answer_relevance(traza: TrazaRAG) -> dict:
    """Evalúa si la respuesta responde a la pregunta (ignora si es correcta o no)."""
    if not traza.respondida:
        return {"relevance_score": None, "reason": "No answer provided"}

    prompt = PROMPT_ANSWER_RELEVANCE.format(
        pregunta=traza.pregunta,
        respuesta=traza.respuesta[:500]
    )
    return completar_json([{"role": "user", "content": prompt}])


def evaluar_traza(traza: TrazaRAG) -> ResultadoEvaluacion:
    """Evalúa una traza completa: faithfulness + context relevance + answer relevance."""

    # Context relevance: media de scores de similitud (no requiere LLM)
    context_relevance = (
        statistics.mean(traza.chunks_scores) if traza.chunks_scores else 0.0
    )

    # Faithfulness: llama al LLM evaluador
    eval_faith = evaluar_faithfulness(traza)

    # Answer relevance: llama al LLM evaluador
    eval_rel   = evaluar_answer_relevance(traza)

    return ResultadoEvaluacion(
        traza_id=traza.id,
        pregunta=traza.pregunta,
        respuesta=traza.respuesta,
        respondida=traza.respondida,
        faithfulness=eval_faith.get("faithfulness_score"),
        context_relevance=round(context_relevance, 4),
        answer_relevance=eval_rel.get("relevance_score"),
        latencia_ms=traza.latencia_ms,
        notas_faithfulness=[
            c["claim"] for c in eval_faith.get("claims", []) if not c.get("supported", True)
        ]
    )


# ── Prueba del evaluador en 3 ejemplos ───────────────────────────────────────
print("Evaluando 3 ejemplos del testset...")
print("(Esto hace ~6 llamadas al LLM evaluador)\n")

evaluaciones_prueba = []
for ejemplo in testset[:3]:
    traza = responder_con_traza(ejemplo.pregunta, coleccion_rag)
    eval_r = evaluar_traza(traza)
    evaluaciones_prueba.append(eval_r)

    print(f"Pregunta:          {ejemplo.pregunta[:60]}...")
    print(f"Respondida:        {eval_r.respondida}")
    print(f"Faithfulness:      {eval_r.faithfulness}")
    print(f"Context relevance: {eval_r.context_relevance}")
    print(f"Answer relevance:  {eval_r.answer_relevance}")
    if eval_r.notas_faithfulness:
        print(f"Afirmaciones no respaldadas: {eval_r.notas_faithfulness}")
    print()

---
## Celda 4 — Evaluación completa y experimento A/B

Corre el testset completo con el pipeline básico y con query rewriting.
Compara las métricas para decidir si el rewriting vale la pena.

In [ ]:
def evaluar_pipeline(
    testset: list[EjemploTest],
    coleccion: chromadb.Collection,
    usar_rewriting: bool = False,
    nombre: str = "pipeline",
    ruta_cache: str | None = None
) -> list[ResultadoEvaluacion]:
    """Evalúa el pipeline en el testset completo. Cachea resultados."""

    if ruta_cache and Path(ruta_cache).exists():
        print(f"[{nombre}] Cargando evaluación desde caché: {ruta_cache}")
        with open(ruta_cache) as f:
            datos = json.load(f)
        return [ResultadoEvaluacion(**d) for d in datos]

    print(f"[{nombre}] Evaluando {len(testset)} preguntas (rewriting={usar_rewriting})...")
    resultados = []

    for i, ejemplo in enumerate(testset):
        print(f"  [{i+1}/{len(testset)}] {ejemplo.pregunta[:50]}...", end=" ")

        traza  = responder_con_traza(
            ejemplo.pregunta, coleccion,
            usar_rewriting=usar_rewriting
        )
        eval_r = evaluar_traza(traza)
        resultados.append(eval_r)

        estado = f"F={eval_r.faithfulness:.2f}" if eval_r.faithfulness is not None else "no-resp"
        print(f"→ {estado}")

    if ruta_cache:
        with open(ruta_cache, "w") as f:
            json.dump([asdict(r) for r in resultados], f, indent=2)
        print(f"  Guardado en '{ruta_cache}'")

    return resultados


def resumen_metricas(resultados: list[ResultadoEvaluacion], nombre: str = "") -> dict:
    """Calcula estadísticas agregadas del conjunto de evaluaciones."""
    respondidas = [r for r in resultados if r.respondida]
    n_total     = len(resultados)
    n_resp      = len(respondidas)

    def media(valores):
        limpios = [v for v in valores if v is not None]
        return round(statistics.mean(limpios), 4) if limpios else None

    return {
        "nombre":              nombre,
        "n_total":             n_total,
        "n_respondidas":       n_resp,
        "tasa_respuesta":      round(n_resp / n_total, 3),
        "faithfulness_media":  media([r.faithfulness for r in respondidas]),
        "context_rel_media":   media([r.context_relevance for r in respondidas]),
        "answer_rel_media":    media([r.answer_relevance for r in respondidas]),
        "latencia_media_ms":   round(statistics.mean([r.latencia_ms for r in resultados]), 1),
        "n_con_alucinaciones": sum(
            1 for r in respondidas
            if r.faithfulness is not None and r.faithfulness < 0.7
        )
    }


# ── Experimento A/B ────────────────────────────────────────────────────────────
# AVISO: esto hace ~4 llamadas al LLM por pregunta (1 retrieval + 1 generación + 2 evaluación)
# Con un testset de 30 preguntas × 2 configuraciones = ~240 llamadas. Coste ~$0.05 con gpt-4o-mini.

# Usa solo las primeras 10 preguntas para la prueba (descomenta todo el testset cuando quieras)
TESTSET_REDUCIDO = testset[:10]

resultados_basico  = evaluar_pipeline(
    TESTSET_REDUCIDO, coleccion_rag,
    usar_rewriting=False,
    nombre="Básico",
    ruta_cache="evaluaciones/resultados_basico.json"
)

resultados_rewrite = evaluar_pipeline(
    TESTSET_REDUCIDO, coleccion_rag,
    usar_rewriting=True,
    nombre="Con rewriting",
    ruta_cache="evaluaciones/resultados_rewrite.json"
)

# ── Tabla comparativa ─────────────────────────────────────────────────────────
res_basico  = resumen_metricas(resultados_basico,  "Básico")
res_rewrite = resumen_metricas(resultados_rewrite, "Con rewriting")

print("\n" + "=" * 62)
print(f"{'MÉTRICA':<30} {'BÁSICO':>12} {'CON REWRITING':>14}")
print("=" * 62)

metricas = [
    ("Tasa de respuesta",      "tasa_respuesta"),
    ("Faithfulness media",     "faithfulness_media"),
    ("Context relevance media","context_rel_media"),
    ("Answer relevance media", "answer_rel_media"),
    ("Latencia media (ms)",    "latencia_media_ms"),
    ("Respuestas con alucinac.","n_con_alucinaciones"),
]

for nombre, clave in metricas:
    v_b = res_basico.get(clave, "—")
    v_r = res_rewrite.get(clave, "—")
    # Indica cuál es mejor (↑ para métricas de calidad, ↓ para latencia y alucinaciones)
    if isinstance(v_b, float) and isinstance(v_r, float):
        if clave in ("latencia_media_ms",):
            indicador = "↑" if v_r < v_b else ("↓" if v_r > v_b else "=")
        elif clave == "n_con_alucinaciones":
            indicador = "✓" if v_r < v_b else ("✗" if v_r > v_b else "=")
        else:
            indicador = "↑" if v_r > v_b else ("↓" if v_r < v_b else "=")
        print(f"{nombre:<30} {str(v_b):>12} {str(v_r):>12} {indicador}")
    else:
        print(f"{nombre:<30} {str(v_b):>12} {str(v_r):>14}")

print("=" * 62)

**Análisis del experimento A/B** (edita esta celda):

- ¿El query rewriting mejoró la faithfulness?
- ¿La latencia extra del rewriting se justifica con la mejora en calidad?
- ¿Qué métrica cambió más? ¿Por qué?
- ¿Qué cambiarías a continuación para mejorar el sistema?

---
## Reto final — Cuadro de mando de evaluación

Construye un detector de preguntas problemáticas que clasifica cada consulta
del testset según su tipo de fallo, para priorizar mejoras.

Lee el esqueleto y completa los `# TODO`.

In [ ]:
from enum import Enum

class TipoFallo(str, Enum):
    SIN_FALLO          = "sin_fallo"           # faithfulness >= 0.9 y respondida
    ALUCINACION        = "alucinacion"          # faithfulness < 0.7 (responde pero inventa)
    FIDELIDAD_PARCIAL  = "fidelidad_parcial"    # 0.7 <= faithfulness < 0.9
    SIN_RESPUESTA      = "sin_respuesta"        # no llegó a responder (sin chunks relevantes)
    BAJA_RELEVANCIA    = "baja_relevancia"      # respondida pero context_relevance < 0.70


def clasificar_fallo(resultado: ResultadoEvaluacion) -> TipoFallo:
    """
    Clasifica el tipo de fallo de una evaluación.

    Prioridad de clasificación:
    1. Si no respondida → SIN_RESPUESTA
    2. Si context_relevance < 0.70 → BAJA_RELEVANCIA
    3. Si faithfulness < 0.70 → ALUCINACION
    4. Si 0.70 <= faithfulness < 0.90 → FIDELIDAD_PARCIAL
    5. Si faithfulness >= 0.90 → SIN_FALLO
    """
    # TODO: implementa la lógica de clasificación
    raise NotImplementedError("Implementa clasificar_fallo()")


def cuadro_de_mando(
    resultados: list[ResultadoEvaluacion],
    nombre_experimento: str = "pipeline"
) -> None:
    """
    Muestra:
    1. Distribución de tipos de fallo
    2. Los 3 peores casos (más urgentes de arreglar)
    3. Recomendaciones de mejora por tipo de fallo
    """
    # TODO: implementa el cuadro de mando
    #
    # Paso 1: clasifica todos los resultados
    # clasificados = [(r, clasificar_fallo(r)) for r in resultados]
    #
    # Paso 2: cuenta por tipo de fallo
    # conteo = Counter(tipo for _, tipo in clasificados)
    #
    # Paso 3: muestra la distribución con porcentajes
    #
    # Paso 4: muestra los 3 casos con faithfulness más baja
    # (solo los respondidos)
    #
    # Paso 5: muestra recomendaciones por tipo de fallo
    # RECOMENDACIONES = {
    #     TipoFallo.ALUCINACION:        "Refuerza la instrucción de grounding en el system prompt",
    #     TipoFallo.FIDELIDAD_PARCIAL:  "Revisa si el prompt permite responder con conocimiento externo",
    #     TipoFallo.SIN_RESPUESTA:      "Baja el umbral de relevancia o aumenta k",
    #     TipoFallo.BAJA_RELEVANCIA:    "Revisa la estrategia de chunking o el modelo de embedding",
    #     TipoFallo.SIN_FALLO:          "Sin acción necesaria",
    # }

    raise NotImplementedError("Implementa cuadro_de_mando()")


# Descomenta cuando hayas implementado las funciones
# print("\n" + "=" * 60)
# print("CUADRO DE MANDO — Pipeline Básico")
# cuadro_de_mando(resultados_basico, "Básico")

# print("\n" + "=" * 60)
# print("CUADRO DE MANDO — Con Query Rewriting")
# cuadro_de_mando(resultados_rewrite, "Con rewriting")

### Extensiones opcionales

1. **RAGAS oficial:** usa la librería `ragas` para replicar el experimento y compara sus scores con los de tu evaluador LLM.
   ```python
   from ragas import evaluate
   from ragas.metrics import faithfulness, answer_relevancy
   from datasets import Dataset
   ```

2. **Umbral óptimo:** experimenta con distintos valores de `UMBRAL_RELEVANCIA` (0.55, 0.60, 0.65, 0.70, 0.75) y grafica la tasa de respuesta frente a la faithfulness media para encontrar el umbral óptimo.

3. **Evaluación automática en CI:** escribe un script Python que cargue el testset, corra el pipeline y falle si la faithfulness media baja de 0.80. Úsalo como gate de calidad antes de desplegar cambios.

In [ ]:
# Extensión 2: búsqueda del umbral óptimo
# Descomenta y ejecuta cuando hayas completado el reto principal

# UMBRALES = [0.55, 0.60, 0.65, 0.70, 0.75]
# resultados_umbral = []

# for umbral in UMBRALES:
#     trazas = [responder_con_traza(e.pregunta, coleccion_rag, umbral=umbral) for e in TESTSET_REDUCIDO]
#     n_resp = sum(1 for t in trazas if t.respondida)
#     resultados_umbral.append({
#         "umbral": umbral,
#         "tasa_respuesta": n_resp / len(trazas),
#     })
#     print(f"Umbral {umbral}: {n_resp}/{len(trazas)} respondidas ({n_resp/len(trazas):.0%})")

# Para graficar:
# import matplotlib.pyplot as plt
# plt.plot([r["umbral"] for r in resultados_umbral],
#          [r["tasa_respuesta"] for r in resultados_umbral], marker="o")
# plt.xlabel("Umbral de relevancia")
# plt.ylabel("Tasa de respuesta")
# plt.title("Tradeoff: umbral de relevancia vs cobertura")
# plt.savefig("evaluaciones/umbral_vs_cobertura.png")
# plt.show()

---
## Autoevaluación de la semana

1. **¿Qué funcionó bien?**  
   ...

2. **¿Qué no entendiste del todo?**  
   ...

3. **¿Qué cambiarías la semana que viene?**  
   ...

4. **Tiempo real dedicado:**  
   ...

---
## Cierre del bloque RAG

Completaste las cuatro semanas del bloque RAG:

- **S05** Embeddings y búsqueda semántica
- **S06** Bases de datos vectoriales y chunking
- **S07** Pipeline RAG completo con query rewriting
- **S08** Evaluación y mejora iterativa ← aquí

La semana 9 empieza el **Bloque 5: Agentes y Tool Use avanzado**.
Construirás el primer agente que decide qué herramientas usar y cuándo parar.

---
*Notebook 08 — Semana 8 del Plan de Formación AI Engineering*